# Caso 2 · Cox07 — Conteos, tasas e intensidad del riesgo
Adaptación didáctica de [A. M. Mayoral, GLM, caso 2](https://asunmayoral.github.io/GLM/caso2/caso2_conteos.html), revisión [25-11-2026](https://github.com/asunmayoral/GLM/tree/2623c88e6633507967a355cd40a8dd403ee90b0f/caso2).

Este cuaderno recorre las siete unidades con explicaciones propias y código R adaptado. No es una transcripción íntegra del material ni sustituye el informe del equipo. Las conclusiones se obtienen al ejecutar los modelos. La tarifa excluye sexo y cualquier variable posterior al periodo que se desea predecir. El sexo se conserva únicamente para diagnóstico de equidad.

## 0 · Preparación
Una fila por póliza en cartera; una por póliza-año en panel; una por parte en siniestros. La unidad de exposición será el año: antiguedad en cartera, expo_anual en panel. Exposicion en el CSV de Cox07 está en décadas. Preservamos el RDS para consultar la verdad del generador al final.

In [ ]:
SEMILLA_CURSO <- 20252026L
set.seed(SEMILLA_CURSO)
options(repr.plot.width=12,repr.plot.height=6)
paquetes <- c("dplyr","tidyr","ggplot2","readr","MASS","glmnet","glmmTMB","DHARMa","car","vcd")
faltan <- setdiff(paquetes,rownames(installed.packages()))
if(length(faltan)) install.packages(faltan,repos="https://cloud.r-project.org")
stopifnot(all(vapply(paquetes,requireNamespace,logical(1),quietly=TRUE)))
library(dplyr)
library(ggplot2)
theme_set(theme_minimal(base_size=14))
base_equipo <- "https://raw.githubusercontent.com/asunmayoral/GLM/b05aeaecde9dca0049ea0ca55fbe162a9b964cbc/caso2/datos_equipos_auto/equipo_07/"
huellas <- c("cartera.csv"="6a8a1cd5a22ec0813be10b4d2173b28f","panel.csv"="813f20b73d15d78bb5777a21268d07c8","siniestros.csv"="9523e2bd2665c492d8dc1fbe5502d7de","cartera.rds"="949486c1f1c5b51ef43f441748e1932d")
archivo <- function(nombre) {
  ruta <- tempfile(fileext=paste0(".",tools::file_ext(nombre)))
  download.file(paste0(base_equipo,nombre),ruta,mode="wb",quiet=TRUE)
  stopifnot(unname(tools::md5sum(ruta))==huellas[[nombre]])
  ruta
}
cartera_rds <- readRDS(archivo("cartera.rds"))
cartera <- readr::read_csv(archivo("cartera.csv"),show_col_types=FALSE)
panel <- readr::read_csv(archivo("panel.csv"),show_col_types=FALSE)
siniestros <- readr::read_csv(archivo("siniestros.csv"),show_col_types=FALSE)
stopifnot(!anyDuplicated(cartera$id_poliza),!anyDuplicated(panel[c("id_poliza","anio")]),sum(cartera$n_siniestros)==nrow(siniestros),sum(panel$n_partes_anual)==nrow(siniestros))
stopifnot(!is.null(attr(cartera_rds,"verdad")),all(cartera$antiguedad>=0),all(panel$expo_anual>=0))
print(summary(cartera$antiguedad-10*cartera$exposicion))
stopifnot(all(cartera$n_siniestros[cartera$antiguedad==0]==0),all(panel$n_partes_anual[panel$expo_anual==0]==0))
cartera <- cartera |> mutate(across(c(id_region,id_agencia,zona_circulacion,tipo_vehiculo,uso,combustible,sexo),factor),edad10=edad_conductor/10,potencia10=potencia_cv/10,km10=km_anuales/10000,veh10=antiguedad_vehiculo/10)
set.seed(SEMILLA_CURSO)
ids <- sample(cartera$id_poliza,floor(.75*nrow(cartera)))
A <- filter(cartera,antiguedad>0,id_poliza %in% ids)
V <- filter(cartera,antiguedad>0,!id_poliza %in% ids)
vars <- c("edad10","potencia10","zona_circulacion","uso","tipo_vehiculo","km10","veh10")
stopifnot(all(complete.cases(cartera[vars])))
f <- function(y,v=vars,e="antiguedad") reformulate(c(v,paste0("offset(log(",e,"))")),y)
valido <- function(m) inherits(m,"glmmTMB") && isTRUE(m$sdr$pdHess) && isTRUE(m$fit$convergence==0)
metricas <- function(y,p) c(MAE=mean(abs(y-p)),RMSE=sqrt(mean((y-p)^2)),sesgo=mean(p-y),observado=sum(y),previsto=sum(p))
print(c(polizas=nrow(cartera),partes=nrow(siniestros),anualidades=nrow(panel),ajuste=nrow(A),reserva=nrow(V)))

## 2.1 · Regresión de Poisson: conteos y tasas
[Unidad original](https://asunmayoral.github.io/GLM/caso2/caso2_conteos.html#sec-u21)

### 1.1 Contexto
El conteo N es entero no negativo. Si el ritmo anual es λ y la exposición es e años, E[N]=eλ. Comparar conteos sin considerar la exposición confunde riesgo con duración. Usamos asistencia para el ejemplo introductorio y siniestros para la tarifa.

In [ ]:
resp <- c("n_asistencia","n_siniestros","n_fraude","n_gestiones")
print(do.call(rbind,lapply(resp,function(y) data.frame(respuesta=y,media=mean(A[[y]]),varianza=var(A[[y]]),ceros=mean(A[[y]]==0),tasa=sum(A[[y]])/sum(A$antiguedad)))))
ggplot(tidyr::pivot_longer(A[c(resp)],everything(),names_to="respuesta",values_to="n"),aes(n))+geom_bar(fill="gold",colour="white")+facet_wrap(~respuesta,scales="free")
brutas <- A |> group_by(zona_circulacion,tipo_vehiculo) |> summarise(partes=sum(n_siniestros),anios=sum(antiguedad),tasa=partes/anios,.groups="drop")
print(brutas)
ggplot(brutas,aes(zona_circulacion,tasa,colour=tipo_vehiculo,group=tipo_vehiculo))+geom_point(size=3)+geom_line()

### 1.2 Modelización y estimación
Poisson: Var(N|x)=μ. El enlace log garantiza una media positiva. η=log(e)+Xβ. El offset tiene coeficiente fijo uno, no es un predictor libre. El score es Xᵀ(y−μ); el peso IWLS es μ.

In [ ]:
m_as <- glm(f("n_asistencia"),data=A,family=poisson())
summary(m_as)
X <- model.matrix(m_as); y <- A$n_asistencia
os <- log(A$antiguedad)
b <- rep(0,ncol(X))
for(i in 1:100) {
  mu <- exp(drop(X%*%b)+os)
  nuevo <- b+solve(crossprod(X,mu*X),crossprod(X,y-mu))
  if(max(abs(nuevo-b))<1e-9) {b<-nuevo;break}
  b<-nuevo
}
print(cbind(IWLS=drop(b),glm=coef(m_as)))

### 1.3 Interpretación y predicción
exp(β) es la razón de tasas (IRR). Multiplica la tasa, no añade partes. Un perfil con exposición de un año tiene media numéricamente igual a su tasa anual. El intervalo de confianza de la media no es el intervalo de predicción de un conteo.

In [ ]:
perfiles <- data.frame(perfil=c("A Joven urbano","B Familiar rural","C Moto urbana","D Furgoneta comercial","E Conductor mayor"),edad10=c(23,45,30,40,70)/10,potencia10=c(110,120,70,130,90)/10,veh10=c(8,6,4,10,15)/10,km10=c(15000,12000,8000,35000,6000)/10000,tipo_vehiculo=c("turismo","turismo","moto","furgoneta","turismo"),zona_circulacion=c("urbana","rural","urbana","mixta","mixta"),uso=c("particular","particular","particular","comercial","particular"),antiguedad=1)
for(v in c("tipo_vehiculo","zona_circulacion","uso")) perfiles[[v]]<-factor(perfiles[[v]],levels=levels(A[[v]]))
print(exp(cbind(IRR=coef(m_as),confint.default(m_as))))
pr <- predict(m_as,perfiles,type="link",se.fit=TRUE)
print(data.frame(perfil=perfiles$perfil,media=exp(pr$fit),ICinf=exp(pr$fit-1.96*pr$se.fit),ICsup=exp(pr$fit+1.96*pr$se.fit),IPinf=qpois(.025,exp(pr$fit)),IPsup=qpois(.975,exp(pr$fit))))

### 1.4–1.5 Inferencia y diagnóstico
Contraste de razón de verosimilitudes (LRT), Wald, AIC y residuos. Si la varianza supera la media condicional, la incertidumbre Poisson puede ser demasiado pequeña. La proporción bruta de ceros no basta para diagnosticar exceso de ceros.

In [ ]:
print(car::Anova(m_as,type=2,test.statistic="LR"))
m0 <- glm(f("n_asistencia",character(0)),data=A,family=poisson())
print(anova(m0,m_as,test="LRT"))
dispersion <- function(m) sum(residuals(m,type="pearson")^2)/df.residual(m)
print(dispersion(m_as))
sim <- DHARMa::simulateResiduals(m_as,n=250)
plot(sim)
print(DHARMa::testDispersion(sim))
print(DHARMa::testZeroInflation(sim))

### 1.6 Validación cruzada y regularización
Lasso penaliza la suma de valores absolutos de los coeficientes; Ridge penaliza la suma de cuadrados; Elastic Net combina ambas. Elegimos λ dentro de ajuste. La reserva no se usa para seleccionar λ. El offset permanece fuera de la penalización.

In [ ]:
m_p <- glm(f("n_siniestros"),data=A,family=poisson())
xx <- model.matrix(reformulate(vars),A)[,-1,drop=FALSE]
xv <- model.matrix(reformulate(vars),V)[,-1,drop=FALSE]
set.seed(SEMILLA_CURSO)
foldid <- sample(rep(1:5,length.out=nrow(A)))
reg <- lapply(c(0,.5,1),function(alfa) glmnet::cv.glmnet(xx,A$n_siniestros,family="poisson",offset=log(A$antiguedad),alpha=alfa,foldid=foldid))
names(reg)<-c("Ridge","ElasticNet","Lasso")
print(sapply(reg,function(m) min(m$cvm)))
plot(reg$Lasso)
print(coef(reg$Lasso,s="lambda.1se"))
print(sapply(reg,function(m) metricas(V$n_siniestros,drop(predict(m,newx=xv,newoffset=log(V$antiguedad),s="lambda.1se",type="response")))))

## 2.2 · Modelos log-lineales para tablas de contingencia
[Unidad original](https://asunmayoral.github.io/GLM/caso2/caso2_conteos.html#sec-u22)

### 2.1 Dos vías
Una tabla de frecuencias es una colección de conteos. El modelo de independencia incluye un efecto por dimensión. El saturado incluye la interacción. La diferencia de devianza permite contrastar independencia. Los residuos muestran qué celdas contribuyen.

In [ ]:
tab <- xtabs(~zona_circulacion+tuvo_siniestro,data=A)
print(tab)
print(chisq.test(tab))
d2 <- as.data.frame(tab)
ind <- glm(Freq~zona_circulacion+tuvo_siniestro,data=d2,family=poisson())
sat <- glm(Freq~zona_circulacion*tuvo_siniestro,data=d2,family=poisson())
print(anova(ind,sat,test="LRT"))
vcd::mosaic(tab,shade=TRUE)
print(transform(d2,esperado=fitted(ind),residuo=residuals(ind,type="pearson")))

### 2.2–2.3 Tres y cuatro vías
El efecto marginal puede esconder asociaciones condicionadas. Comparamos jerarquías de interacciones: independencia, asociaciones por pares y saturación. El logit de tener parte es una lectura complementaria; no equivale a una tasa anual si las exposiciones difieren.

In [ ]:
d3 <- as.data.frame(xtabs(~zona_circulacion+tipo_vehiculo+tuvo_siniestro,data=A))
m3i <- glm(Freq~zona_circulacion+tipo_vehiculo+tuvo_siniestro,data=d3,family=poisson())
m3p <- glm(Freq~(zona_circulacion+tipo_vehiculo+tuvo_siniestro)^2,data=d3,family=poisson())
m3s <- glm(Freq~zona_circulacion*tipo_vehiculo*tuvo_siniestro,data=d3,family=poisson())
print(anova(m3i,m3p,m3s,test="LRT"))
print(transform(d3,esperado=fitted(m3p)))
print(sum(fitted(m3p)<5))
print(summary(glm(tuvo_siniestro~zona_circulacion*tipo_vehiculo,data=A,family=binomial())))
d4 <- as.data.frame(xtabs(~zona_circulacion+tipo_vehiculo+uso+tuvo_siniestro,data=A))
m4i <- glm(Freq~zona_circulacion+tipo_vehiculo+uso+tuvo_siniestro,data=d4,family=poisson())
m4p <- glm(Freq~(zona_circulacion+tipo_vehiculo+uso+tuvo_siniestro)^2,data=d4,family=poisson())
print(anova(m4i,m4p,test="LRT"))
print(sum(fitted(m4p)<5))

### 2.4 Tablas de cambio: bonus-malus
Simetría significa que el flujo i→j es igual al flujo j→i. Homogeneidad marginal significa que la distribución antes y después coincide; es una condición menos restrictiva. La cuasisimetría permite efectos marginales distintos. Los ceros estructurales deben distinguirse de los ceros muestrales. Excluimos las pólizas cuyo último año no dispone de nivel previo independiente.

In [ ]:
bm <- cartera |> left_join(panel |> group_by(id_poliza) |> summarise(ultimo=max(anio),.groups="drop"),by="id_poliza") |> filter(ultimo>1,baja==0)
lev <- as.character(1:5)
bm$prev <- factor(bm$bonus_malus_prev,levels=lev)
bm$act <- factor(bm$bonus_malus_act,levels=lev)
tb <- table(bm$prev,bm$act)
print(tb)
d <- as.data.frame(tb);names(d)<-c("prev","act","Freq")
d$i<-as.integer(d$prev);d$j<-as.integer(d$act)
d$par<-factor(paste(pmin(d$i,d$j),pmax(d$i,d$j),sep=":"))
d$diagonal<-as.integer(d$i==d$j)
d$sube1<-as.integer(d$j==d$i+1)
ggplot(d,aes(prev,act,fill=Freq))+geom_tile()+geom_text(aes(label=Freq))+scale_fill_gradient(low="white",high="goldenrod")
# Bowker: pares discordantes con denominador positivo.
z <- upper.tri(tb)&(tb+t(tb)>0)
bowker <- sum((tb-t(tb))[z]^2/(tb+t(tb))[z])
print(c(Bowker=bowker,gl=sum(z),p=pchisq(bowker,sum(z),lower.tail=FALSE)))
# Stuart-Maxwell: covarianza de diferencias marginales, rango efectivo.
delta <- rowSums(tb)-colSums(tb)
S <- diag(rowSums(tb)+colSums(tb))-tb-t(tb)
gl <- qr(S)$rank
if(gl>0){sm<-drop(t(delta)%*%MASS::ginv(S)%*%delta);print(c(StuartMaxwell=sm,gl=gl,p=pchisq(sm,gl,lower.tail=FALSE)))}
simetria <- glm(Freq~par,data=d,family=poisson())
cuasi <- try(glm(Freq~prev+act+par,data=d,family=poisson()),silent=TRUE)
mov <- glm(Freq~prev+act+I(i*j)+diagonal,data=d,family=poisson())
print(summary(mov))
if(inherits(cuasi,"glm") && cuasi$converged) print(anova(simetria,cuasi,test="LRT"))

#### Proyección de renovaciones
Usamos probabilidades empíricas por fila como referencia transparente; no asumimos que un modelo log-lineal mal ajustado sea mejor. Proyectamos cartera viva con crecimiento neto de 10 %, sin bajas adicionales en este escenario. Los nuevos entran en nivel 3 después de las transiciones. Simulamos incertidumbre de transición con Dirichlet y variación multinomial; la suavización 0.5 es una decisión explícita y puede asignar masa a transiciones no observadas. Revisar reglas de bonus-malus antes de usarla como presupuesto definitivo.

In [ ]:
P <- sweep(tb,1,rowSums(tb),"/")
print(P)
actual <- as.numeric(table(factor(cartera$bonus_malus_act[cartera$baja==0],levels=lev)))
nuevos <- round(.1*sum(actual))
set.seed(SEMILLA_CURSO)
simBM <- replicate(2000,{
  suma<-numeric(5)
  for(i in 1:5){
    z<-rgamma(5,shape=as.numeric(tb[i,])+.5)
    suma<-suma+drop(rmultinom(1,actual[i],z/sum(z)))
  }
  suma[3]<-suma[3]+nuevos
  suma
})
print(data.frame(nivel=lev,actual=actual,previsto=rowMeans(simBM),inf=apply(simBM,1,quantile,.025),sup=apply(simBM,1,quantile,.975)))

## 2.3 · Sobredispersión
[Unidad original](https://asunmayoral.github.io/GLM/caso2/caso2_conteos.html#sec-u23)

### 3.1–3.2 Diagnóstico y estructuras de varianza
Poisson: V=μ. Quasi-Poisson: V=φμ. NB1: V=μ(1+φ). NB2: V=μ+μ²/θ. La heterogeneidad entre clientes puede producir sobredispersión; no implica automáticamente inflación de ceros.

In [ ]:
print(dispersion(m_p))
nube <- A |> mutate(mu=fitted(m_p),grupo=ntile(mu,10)) |> group_by(grupo) |> summarise(media=mean(mu),res2=mean((n_siniestros-mu)^2),.groups="drop")
ggplot(nube,aes(media,res2))+geom_point(size=3)+geom_abline(slope=1,intercept=0,colour="tomato")
rootograma <- function(y,prob,K=8){
  obs<-tabulate(pmin(y,K)+1,nbins=K+1)
  esp<-colSums(prob)
  ggplot(data.frame(k=0:K,r=sqrt(obs)-sqrt(esp)),aes(k,r))+geom_col(fill="goldenrod")+geom_hline(yintercept=0)+labs(y="√observado − √esperado",x="Conteo (último = cola)")
}
prob_p <- function(mu,K=8) cbind(sapply(0:(K-1),function(k)dpois(k,mu)),ppois(K-1,mu,lower.tail=FALSE))
rootograma(A$n_siniestros,prob_p(fitted(m_p)))

### 3.3 Quasi-Poisson
Corrige la varianza de los coeficientes sin cambiar la media ajustada. No dispone de verosimilitud completa: no comparar su AIC con Poisson o NB.

In [ ]:
m_q <- glm(f("n_siniestros"),data=A,family=quasipoisson())
print(summary(m_q))
print(cbind(Poisson=coef(summary(m_p))[,2],Quasi=coef(summary(m_q))[,2]))
print(max(abs(fitted(m_p)-fitted(m_q))))

### 3.4–3.5 Binomial negativa y comparación
La mezcla Poisson–Gamma produce NB2. θ controla la heterogeneidad: cuando θ crece, el modelo converge a Poisson. Comparamos también NB1. Los intervalos predictivos deben incluir el ruido del conteo, no solo la incertidumbre de β.

In [ ]:
m_nb <- MASS::glm.nb(f("n_siniestros"),data=A)
m_nb1 <- glmmTMB::glmmTMB(f("n_siniestros"),data=A,family=glmmTMB::nbinom1())
print(AIC(m_p,m_nb,m_nb1))
print(summary(m_nb))
print(m_nb$theta)
prob_nb <- function(mu,theta,K=8) cbind(sapply(0:(K-1),function(k)dnbinom(k,mu=mu,size=theta)),pnbinom(K-1,mu=mu,size=theta,lower.tail=FALSE))
rootograma(A$n_siniestros,prob_nb(fitted(m_nb),m_nb$theta))
plot(DHARMa::simulateResiduals(m_nb,n=250))
print(rbind(Poisson=metricas(V$n_siniestros,predict(m_p,V,type="response")),NB2=metricas(V$n_siniestros,predict(m_nb,V,type="response")),NB1=metricas(V$n_siniestros,predict(m_nb1,V,type="response"))))
pr<-predict(m_nb,perfiles,type="link",se.fit=TRUE)
print(data.frame(perfil=perfiles$perfil,media=exp(pr$fit),ICinf=exp(pr$fit-1.96*pr$se.fit),ICsup=exp(pr$fit+1.96*pr$se.fit),IPinf=qnbinom(.025,mu=exp(pr$fit),size=m_nb$theta),IPsup=qnbinom(.975,mu=exp(pr$fit),size=m_nb$theta)))

## 2.4 · Demasiados ceros
[Unidad original](https://asunmayoral.github.io/GLM/caso2/caso2_conteos.html#sec-u24)

### 4.1–4.2 Diagnóstico
Usamos n_fraude para estudiar el fenómeno. Primero calculamos ceros esperados bajo Poisson y NB. Un exceso aparente puede desaparecer al reconocer heterogeneidad o exposición. No presuponemos que el fraude tenga ceros estructurales.

In [ ]:
ff <- f("n_fraude")
fp <- glm(ff,data=A,family=poisson())
fnb <- MASS::glm.nb(ff,data=A)
print(c(observados=sum(A$n_fraude==0),Poisson=sum(dpois(0,fitted(fp))),NB=sum(dnbinom(0,mu=fitted(fnb),size=fnb$theta))))
print(DHARMa::testZeroInflation(DHARMa::simulateResiduals(fp,n=250)))
print(DHARMa::testZeroInflation(DHARMa::simulateResiduals(fnb,n=250)))

### 4.3 Zero-inflated
Un ZIP mezcla un cero estructural con Poisson. P(N=0)=π+(1−π)exp(−μ); E[N]=(1−π)μ. ZINB sustituye Poisson por NB. El grupo estructural es una hipótesis latente, no una etiqueta observada.

In [ ]:
fz <- ~zona_circulacion+uso
zip <- glmmTMB::glmmTMB(ff,data=A,ziformula=fz,family=poisson())
zinb <- glmmTMB::glmmTMB(ff,data=A,ziformula=fz,family=glmmTMB::nbinom2())
print(summary(zip))
print(AIC(fp,fnb,zip,zinb))
print(c(ZIP=valido(zip),ZINB=valido(zinb)))
if(valido(zip)) {
  pi0<-predict(zip,type="zprob")
  mu<-predict(zip,type="conditional")
  print(c(ceros_esperados=sum(pi0+(1-pi0)*exp(-mu)),media=sum((1-pi0)*mu)))
  plot(DHARMa::simulateResiduals(zip,n=250))
}

### 4.4 Hurdle
La barrera separa cero de positivo. El conteo positivo usa una distribución truncada en cero. No es lo mismo que añadir una masa a una Poisson ordinaria. En glmmTMB, el componente cero y la familia truncada construyen el hurdle.

In [ ]:
hp <- glmmTMB::glmmTMB(ff,data=A,ziformula=fz,family=glmmTMB::truncated_poisson())
hnb <- glmmTMB::glmmTMB(ff,data=A,ziformula=fz,family=glmmTMB::truncated_nbinom2())
print(c(HurdleP=valido(hp),HurdleNB=valido(hnb)))
if(valido(hp)) plot(DHARMa::simulateResiduals(hp,n=250))

### 4.5 Comparación y conexión con la tarifa
Solo consideramos ajustes convergentes. Evaluamos conteos de reserva y AIC. La evidencia en fraude no autoriza automáticamente trasladar el modelo al número de siniestros. Repetimos la comparación para la respuesta de tarificación.

In [ ]:
mods_f<-list(Poisson=fp,NB=fnb,ZIP=zip,ZINB=zinb,HurdleP=hp,HurdleNB=hnb)
mods_f<-Filter(function(m) !inherits(m,"glmmTMB")||valido(m),mods_f)
print(sapply(mods_f,AIC))
print(sapply(mods_f,function(m) metricas(V$n_fraude,predict(m,V,type="response"))))
claim_zi<-glmmTMB::glmmTMB(f("n_siniestros"),data=A,ziformula=~1,family=glmmTMB::nbinom2())
claim_h<-glmmTMB::glmmTMB(f("n_siniestros"),data=A,ziformula=~1,family=glmmTMB::truncated_nbinom2())
mods_claim<-Filter(function(m) !inherits(m,"glmmTMB")||valido(m),list(Poisson=m_p,NB=m_nb,NB1=m_nb1,ZINB=claim_zi,Hurdle=claim_h))
print(sapply(mods_claim,AIC))
print(sapply(mods_claim,function(m) metricas(V$n_siniestros,predict(m,V,type="response"))))

## 2.5 · Efectos aleatorios y modelos mixtos
[Unidad original](https://asunmayoral.github.io/GLM/caso2/caso2_conteos.html#sec-u25)

### 5.1–5.2 Agrupamiento y anidamiento
Las pólizas comparten agencia y región. Usamos gestiones para estudiar estructura territorial, después siniestros para la tarifa. El modelo (1|region/agencia) contiene intercepto regional y de agencia dentro de región.

In [ ]:
territorio <- A |> group_by(id_region,id_agencia) |> summarise(n=n(),tasa=sum(n_gestiones)/sum(antiguedad),.groups="drop")
ggplot(territorio,aes(reorder(id_agencia,tasa),tasa,colour=id_region))+geom_point(size=3)+coord_flip()
fg <- f("n_gestiones")
g0 <- glm(fg,data=A,family=poisson())
g1 <- glmmTMB::glmmTMB(update(fg,.~.+(1|id_agencia)),data=A,family=poisson())
g2 <- glmmTMB::glmmTMB(update(fg,.~.+(1|id_region/id_agencia)),data=A,family=poisson())
print(AIC(g0,g1,g2))
print(glmmTMB::VarCorr(g2))

### 5.3 Interpretación y predicción
El efecto aleatorio modifica la tasa multiplicativamente. Una agencia típica fija u=0. La media de agencias nuevas integra la normal de efectos: exp(η+σ²/2). La predicción condicional sustituye el efecto estimado; no integra automáticamente la incertidumbre posterior.

In [ ]:
if(valido(g2)){
  re<-glmmTMB::ranef(g2,condVar=TRUE)$cond
  print(re)
  eta<-predict(g2,V,type="link",re.form=NA)
  vv<-sum(vapply(glmmTMB::VarCorr(g2)$cond,function(x)x[1,1],numeric(1)))
  print(rbind(tipica=metricas(V$n_gestiones,exp(eta)),marginal=metricas(V$n_gestiones,exp(eta+vv/2)),condicional=metricas(V$n_gestiones,predict(g2,V,type="response"))))
  set.seed(SEMILLA_CURSO)
  mu_nueva<-exp(log(1)+mean(eta-log(V$antiguedad))+rnorm(4000,0,sqrt(vv)))
  print(quantile(rpois(4000,mu_nueva),c(.025,.5,.975)))
}

### 5.4–5.5 Selección y diagnóstico
Comparamos Poisson mixto y NB mixto. Evitamos decidir por AIC si el Hessiano no es positivo. Para siniestros examinamos si la región o la agencia añaden información tras ajustar el perfil.

In [ ]:
gnb <- glmmTMB::glmmTMB(update(fg,.~.+(1|id_region/id_agencia)),data=A,family=glmmTMB::nbinom2())
print(AIC(g2,gnb))
if(valido(g2)) plot(DHARMa::simulateResiduals(g2,n=250))
red <- glmmTMB::glmmTMB(update(f("n_siniestros"),.~.+(1|id_region/id_agencia)),data=A,family=glmmTMB::nbinom2())
if(valido(red)){
  print(glmmTMB::VarCorr(red))
  print(metricas(V$n_siniestros,predict(red,V,type="response")))
  print(glmmTMB::ranef(red)$cond)
  plot(DHARMa::simulateResiduals(red,n=250))
}

### 5.6–5.7 OLRE y validación
Un efecto aleatorio por observación introduce heterogeneidad Poisson-lognormal. No es equivalente a NB2 ni prueba que exista una causa individual persistente. El efecto por póliza se estudia longitudinalmente en la siguiente unidad. La verdad del DGP se abre al final.

In [ ]:
A$id_obs<-factor(A$id_poliza)
olre<-glmmTMB::glmmTMB(update(f("n_siniestros"),.~.+(1|id_obs)),data=A,family=poisson())
print(c(OLRE=valido(olre),NB=TRUE))
if(valido(olre)) print(AIC(olre,m_nb))

## 2.6 · Riesgos a trozos
[Unidad original](https://asunmayoral.github.io/GLM/caso2/caso2_conteos.html#sec-u26)

### 6.1–6.3 Panel y exploración
Cada fila representa tiempo cubierto en un año de antigüedad. Sumamos partes y exposición, no promediamos tasas individuales. Dividimos a priori en años 1, 2, 3, 4 y 5+. Revisamos tamaño y tasa de cada tramo. Las covariables de ficha se mantienen constantes para estudiar el efecto de la antigüedad; no confundimos edad cambiante con etapa de póliza.

In [ ]:
p <- panel |> filter(expo_anual>0) |> left_join(cartera |> select(id_poliza,all_of(vars),id_region,id_agencia),by="id_poliza") |> mutate(etapa=factor(pmin(anio,5),levels=1:5,labels=c("1","2","3","4","5+")))
PA <- filter(p,id_poliza %in% ids)
PV <- filter(p,!id_poliza %in% ids)
por_anio <- PA |> group_by(anio) |> summarise(partes=sum(n_partes_anual),expo=sum(expo_anual),tasa=partes/expo,.groups="drop")
print(por_anio)
ggplot(por_anio,aes(anio,tasa,size=expo))+geom_line()+geom_point(colour="tomato")

### 6.4–6.5 Ajuste y dependencia
El modelo constante impone la misma tasa en todos los años. El modelo a trozos permite cambios. El intercepto aleatorio por póliza mide heterogeneidad persistente. Una interacción con etapa pregunta si un predictor modifica el riesgo de modo distinto a lo largo del tiempo.

In [ ]:
fa<-f("n_partes_anual",vars,"expo_anual")
fe<-update(fa,.~.+etapa)
const<-glm(fa,data=PA,family=poisson())
pw<-glm(fe,data=PA,family=poisson())
print(anova(const,pw,test="LRT"))
pw_mix<-glmmTMB::glmmTMB(update(fe,.~.+(1|id_poliza)),data=PA,family=poisson())
pw_nb<-glmmTMB::glmmTMB(update(fe,.~.+(1|id_poliza)),data=PA,family=glmmTMB::nbinom2())
print(AIC(const,pw,pw_mix,pw_nb))
inter<-update(pw,.~.+etapa:edad10+etapa:zona_circulacion)
print(anova(pw,inter,test="LRT"))

### 6.6 Diagnóstico y predicción
Para pólizas nuevas no conocemos su efecto aleatorio. Integramos la distribución para obtener la tasa media y simulamos el conteo futuro. Para clientes conocidos, usamos únicamente el historial anterior al año que deseamos predecir.

In [ ]:
if(valido(pw_mix)){
  sim<-DHARMa::simulateResiduals(pw_mix,n=250)
  plot(sim)
  plot(DHARMa::recalculateResiduals(sim,group=PA$id_poliza))
  v<-glmmTMB::VarCorr(pw_mix)$cond$id_poliza[1,1]
  mu<-predict(pw_mix,PV,type="response",re.form=NA,allow.new.levels=TRUE)*exp(v/2)
  print(metricas(PV$n_partes_anual,mu))
  pf<-merge(perfiles,data.frame(etapa=levels(PA$etapa)))
  pf$etapa<-factor(pf$etapa,levels=levels(PA$etapa));pf$expo_anual<-1
  pf$id_poliza<-PA$id_poliza[1]
  eta<-predict(pw_mix,pf,type="link",re.form=NA)
  pf$tasa<-exp(eta+v/2)
  set.seed(SEMILLA_CURSO)
  ip<-t(vapply(eta,function(e) quantile(rpois(4000,exp(e+rnorm(4000,0,sqrt(v)))),c(.025,.975)),numeric(2)))
  pf$inf<-ip[,1];pf$sup<-ip[,2]
  print(pf[c("perfil","etapa","tasa","inf","sup")])
  print(ggplot(pf,aes(etapa,tasa,group=perfil,colour=perfil))+geom_line()+geom_point())
}
p <- p |> group_by(id_poliza) |> arrange(anio,.by_group=TRUE) |> mutate(previos=lag(cumsum(n_partes_anual),default=0),expo_previa=lag(cumsum(expo_anual),default=0),ultimo=anio==max(anio),varios=n()>1) |> ungroup()
train <- filter(p,!ultimo | !varios)
test <- filter(p,ultimo,varios)
ren <- glmmTMB::glmmTMB(update(fe,.~.+(1|id_poliza)),data=train,family=poisson())
if(valido(ren)){
  print(rbind(historial=metricas(test$n_partes_anual,predict(ren,test,type="response")),tipico=metricas(test$n_partes_anual,predict(ren,test,type="response",re.form=NA))))
}
print(test |> group_by(previos=pmin(previos,3)) |> summarise(partes=sum(n_partes_anual),expo=sum(expo_anual),tasa=partes/expo,.groups="drop"))

### 6.7 Verificación contra el DGP
Abrimos la verdad solo después de especificar y evaluar los modelos. El DGP de auto usa frailty, riesgo Weibull y efectos territoriales; no todos los parámetros son directamente comparables con un GLM agregado. Revisamos la escala antes de contrastarlos. La prueba de equidad usa sexo solo como diagnóstico, nunca como predictor de tarifa.

In [ ]:
verdad <- attr(cartera_rds,"verdad")
print(names(verdad))
print(verdad[c("theta_frail","weib_shape","lambda0","sigma_reg","sigma_ag")])
print(verdad$betas$claim)
print(coef(m_nb))
print(verdad$proxy)
print(prop.table(table(A$sexo,A$tipo_vehiculo),1))
print(V |> mutate(pred=predict(m_nb,V,type="response")) |> group_by(sexo) |> summarise(observado=sum(n_siniestros)/sum(antiguedad),previsto=sum(pred)/sum(antiguedad),.groups="drop"))

## 2.7 · Estudio de caso, exposición y evaluación
[Encargo original](https://asunmayoral.github.io/GLM/caso2/caso2_conteos.html#sec-u27)

El objetivo es una tarifa anual de frecuencia explicable y validada. Las cinco preguntas conectan las unidades:
- P1: zona y vehículo, asociación condicionada y tamaños de celda (2.2).
- P2: selección de predictores y familia; tarifa de cinco perfiles e intervalos (2.1, 2.3, 2.4).
- P3: red territorial, agencias y predicción para una nueva agencia (2.5).
- P4: transición bonus-malus, crecimiento y margen (2.2).
- P5: renovación, etapas de antigüedad e historial previo (2.6).

### Resumen de modelos
| Familia | Media | Varianza o estructura |
|---|---|---|
| Poisson | μ | μ |
| Quasi-Poisson | μ | φμ |
| NB1 | μ | μ(1+φ) |
| NB2 | μ | μ+μ²/θ |
| ZIP/ZINB | (1−π)μ | mezcla con cero |
| Hurdle | P(N>0) E[N|N>0] | distribución truncada |
| GLMM | μ(u) | dependencia por grupo |

El informe ejecutivo tiene como destinatario al comité; reservar las fórmulas y diagnósticos para el anexo. Elegir el modelo definitivo con evidencia, no por número de modelos probados. Este cuaderno es una edición de estudio, no la reproducción literal de los scripts ni del texto. [Scripts originales](https://github.com/asunmayoral/GLM/tree/2623c88e6633507967a355cd40a8dd403ee90b0f/caso2/scripts). No se ha ejecutado de principio a fin.

In [ ]:
sessionInfo()